In [ ]:
from ngsolve import *
from ngsolve.webgui import Draw
from netgen.occ import *
from netgen.geom2d import SplineGeometry
import numpy as np
from math import sqrt
from ngsolve import x, y, sin, sinh, cosh, pi

import inspect

       
def gravitationsEinfluss_Auflager(l,b,t,step,maxh,order):
    #region GEOMETRIE
    shape = MoveTo(0,0).Rectangle(l,b).Face()
    shape.edges.name="dirichlet"
    geo = OCCGeometry(shape,dim=2)

    mesh = Mesh(geo.GenerateMesh(maxh=maxh))
    mesh.Curve(3)
    print("geometry - done")
    #Draw(mesh)
    #endregion

    #region MATERIALKONSTANTEN
    E  = 70e6      #Glas ~ N'/mm² Elastizitätsmodul - ACHTUNG: N' nicht gleich N!!!      
    """
    UMRECHNUNG (m -> mm)

    Einheit Pascal: 1 Pa = 1 N/m² 
    Einheit Newton: 1 N = 1 kg*m/s²     - dh müssen auch Newton umrechnen

    Definiere:
    N' := kg*mm/s²      - in Millimeter!!!!
        -> 1 N = 1000 N'

    1 Pa = 1 N/m² = 10e-6 N/mm² = 10e-6 * 10e3 N'/mm² = 10e-3 N'/mm²

    Glas hat E-Modul von 70e9 Pa (70 GPa) = 70e9 N/m² = 70e9 * 10^(-3) N'/mm² = 70e6 N'/mm²         #neuer Wert
    """
    nu = 0       #dimensionslos, also bei m und mm gleich
    rho = 2.5e-6   #kg/mm³ Dichte 
    """
    Dichte von Glas 2500 kg/m³ = 2.5 * 10e3 kg/m³
    1kg/m³ = 1000g/10e9mm³ = 10e(-6)g/mm³
    -> 2.5 * 10e3 * 10e(-6) g/mm³ = 2.5 * 10e(-3) g/mm³ = 0.0025 g/mm³

    """

    g = 9810     # Erdbeschleunigung mm/s²
    """
    Erdbeschleunigung g = 9.81 m/s² = 9810 mm/s²
    """   
    f = CF(4*pi**4*sin(pi*x)*sin(pi*y))
    q = rho * t * g     #Eigengewicht der Platte punktweise!!! - rechte Seite der PDE, also unser f_PDE
    print("materialkonstanten - done")
    #endregion

    #region FUNKTIONEN für PDE
    # Db = E*t**3/(12*(1-nu**2))
    Db=1

    def D(A):
        return Db *((1-nu)*A+ nu*Trace(A)*Id(2))

    def Dinv(A):
        return 1/Db * (1/(1-nu)*A - nu/(1-nu**2)*Trace(A)*Id(2))
    print("funktionen deklarieren - done")
    #endregion

    #region SIMULATION: FUNKTIONENRAUM,SCHWACHE FORM,VISUALISIERUNG
    V = HDivDiv(mesh, order=order-1,dirichlet="dirichlet")
    Q = H1(mesh, order=order, dirichlet="dirichlet")
    X = V * Q

    (sigma,w),(tau,v)= X.TnT()

    n = specialcf.normal(2)

    def tang(u):
        return u - (u*n)*n

    a = BilinearForm(X,symmetric=True)
    a += InnerProduct(Dinv(sigma),tau) * dx
    a += div(sigma)*Grad(v) * dx
    a += div(tau) * Grad(w) * dx
    a += - (sigma[n,:] * tang(Grad(v)) + tau[n,:] * tang(Grad(w))  ) * dx(element_boundary=True)
    a.Assemble()

    L = LinearForm(X)
    L += f * v *dx
    L.Assemble()
    print("a und L assemble - done")

    gf_solution = GridFunction(X)
    gf_solution.vec.data = a.mat.Inverse(X.FreeDofs(),inverse="") * L.vec

    gf_sigma, gf_w = gf_solution.components
    #Draw(gf_sigma, mesh,name="sigma")
    # Draw(100*gf_w, mesh, name="disp",deformation=True, euler_angles=[-60,5,30])
    print("simulation - done")
    #endregion

    #region L2-Norm
    #analytische Lösung:
    den = 5 + 8*pi**2 + 3*cosh(4*pi)

    a = -2*(sinh(pi) - 3*sinh(3*pi) + pi*(4*pi*sinh(pi) + 7*cosh(pi) - 3*cosh(3*pi))) / den
    b = -8*pi*(2*pi*sinh(pi) + cosh(pi)) / den
    c = (10*cosh(pi) + 6*cosh(3*pi) + 16*pi*(sinh(pi) + pi*cosh(pi))) / den
    d = 2*pi*(5*sinh(pi)  - 3*sinh(3*pi)  + 4*pi*cosh(pi)) / den

    u_ref = (
        ((a + b*x)*cosh(pi*x)
        + (c + d*x)*sinh(pi*x)
        + sin(pi*x))
        * sin(pi*y)
    )

    error = sqrt(Integrate((gf_w - u_ref)**2, mesh))
    print("L2-Fehler =", error)

    # max_error = max(abs(gf_w(mesh(x,y))-u_ref(mesh(x,y))))
    #endregion

    # returnWert ist egl nicht wichtig, weil wir uns den dateinamen eh aus den anderen variablen bestimmen können
    return error


def calc_L2norm_abstand(filename_ref,maxh, step, order, mass_Omega, function):
    """
    wir berechnen die L2norm über ein funktionenraster, dabei erhalten wir folgende formel
    ||uh-uref||_2 = sqrt( sum_{i=1}^{anzahl_punkte} (u_h(i)-u_ref(i))^2 * maß(omega)/anzahl_pkt)
    = sqrt( summe[ (uh(i)-uref(i))^2 ] * maß_Omega / anzahl_pkte )
    """

    filename_func = f"xyzFiles/h{maxh}_s{step}_o{order}_{function}.xyz"
    #x_uh = data_uh[:,0]    
    #y_uh = data_uh[:,1]
    z_uh = filename_func[:,2]

    data_u_ref = np.loadtxt(filename_ref)
    #x_u_ref = data_u_ref[:,0]    
    #y_u_ref = data_u_ref[:,1]
    z_u_ref = data_u_ref[:,2]

    normierungsZahl = step*step
    sum = 0
    maxDiff = 0

    for i in range(normierungsZahl):
        diff = abs(z_uh[i]-z_u_ref[i])
        #berechnen Maximalabstand
        if diff > maxDiff:
            maxDiff = diff

        #quadrierter summand für integral-summe
        sum += diff*diff
    integral = sum/normierungsZahl*mass_Omega
    L2_norm = sqrt(integral)
    print("L2norm - done")
    return L2_norm, maxDiff


if __name__ == "__main__":
    l = 2000
    b = 1000
    t = 5
    step = 40
    order = 3
    mass_Omega = l*b
    print("gestartet!")
    maxh_array = [100,5,10,50,1,200]

    for maxh in maxh_array:
        print(maxh)
        error, max_error = gravitationsEinfluss_Auflager(l,b,t,step,maxh,order)
        print("für maxh=",maxh": \nL2-Norm-Fehler: ",error,"\nmax Fehler: ",max_error,"\wobei order=",order," und step=",step)


KeyboardInterrupt: 